# ADNI Tabular Survival Analysis Pipeline

This notebook trains, evaluates, and interprets **four survival models** on ADNI tabular data
to predict the time until a patient converts from one cognitive state to the next.

| # | Model | Library | Strengths |
|---|-------|---------|-----------|
| 1 | **Cox PH** | lifelines | Interpretable coefficients, well-calibrated, fast |
| 2 | **GBSA** | scikit-survival | Handles non-linearity and feature interactions |
| 3 | **DeepSurv** | pycox / torchtuples | Deep representations, flexible architecture |
| 4 | **Ensemble** | custom | Combines strengths of models 1–3 |

---

### How overfitting is prevented

| Safeguard | Where applied |
|-----------|---------------|
| 20 % stratified **holdout** locked away before any training | Section 5 |
| **OOF (out-of-fold)** C-index used for all HPO decisions | Sections 6a–6c |
| **Train vs. OOF gap** printed for every model | Sections 6a–6c |
| **Bootstrap CI** on holdout gives honest uncertainty estimate | Sections 7, 9j |
| **Early stopping** (patience=10) in every DeepSurv CV fold | Section 6c |
| Regularisation penalty in CoxPH prevents coefficient collapse | Section 6a |

---

### Key metrics used

- **Harrell's C-index** — probability that the model ranks two random subjects correctly
  (1.0 = perfect, 0.5 = random coin flip).
- **IPCW Antolini C-td** — time-dependent version that accounts for censoring via
  inverse-probability-of-censoring weighting (more appropriate than Harrell when
  follow-up times vary widely).
- **Integrated Brier Score (IBS)** — mean squared prediction error over time
  (0 = perfect, 0.25 = uninformative, lower is better).
- **AUC at fixed horizons** — standard ROC area for "will event occur within *t* years?".


## 0 · Imports & configuration

Running this cell loads all modules and creates the output directories.
Nothing is trained here — it is safe to re-run at any time.

> **`RETRAIN = True`** trains from scratch and saves checkpoints.  
> **`RETRAIN = False`** loads previously saved checkpoints, skipping expensive HPO.


In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 120

# ── project path ──────────────────────────────────────────────────────────────
BASE = Path(".")  # run from the notebook's own directory
sys.path.insert(0, str(BASE))

from config import (
    RANDOM_SEED, N_FOLDS, HORIZONS,
    FIG_DIR, CHECKPOINT_DIR, OUT_DIR, BASE_DIR,
    MRI_HARMONIZE_COLS,
)
from preprocessing import (
    build_survival_labels, run_combat, plot_before_after, harmonization_report,
    compute_slopes_cutoff, longitudinal_fill, mice_impute, assemble_cohort,
    add_slope_concordance, get_domain_features, classify_reverters,
)
from modeling import (
    make_holdout_split,
    bootstrap_cindex_harrell, bootstrap_cindex_td,
    permutation_importance_survival,
    gbsa_survival_cv, run_deepsurv, run_cox_ph,
    calc_gbsa_c, calc_deepsurv_c, calc_cox_ph_c,
    weighted_ensemble_td,
    horizon_aucs,
    save_checkpoint, load_checkpoint,
    build_csf_imputer,
)
from postprocessing import (
    km_risk_quartile, km_binary_comparison,
    plot_individual_survival_curves, plot_median_survival_by_group,
    plot_feature_importance, plot_cox_coefficients, plot_permutation_importance,
    plot_learning_curve_deepsurv,
    plot_overfitting_summary,
    plot_bootstrap_ci, plot_bootstrap_distribution,
    plot_roc_horizons,
    plot_brier_score, plot_calibration_horizon,
    plot_model_comparison,
    plot_optuna_history,
    plot_survival_curve_grid,
)

for d in [FIG_DIR, CHECKPOINT_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RETRAIN = True  # set False to load saved checkpoints
print("Setup complete.")


## 1 · Load & clean data

Replace the placeholder block below with your actual data loading.  
The pipeline requires exactly two DataFrames:

| Variable | Description | Required columns |
|----------|-------------|-----------------|
| `df_all` | Full longitudinal table — one row per subject × visit | `RID`, `Years_bl`, `DX`, `VISCODE` |
| `df_baseline` | Baseline visit only — one row per subject | `RID`, `DX_bl`, `AGE`, `PTGENDER`, `APOE4`, `FLDSTRENG` |

**Expected output:** a print confirming row counts, e.g.  
`df_all: 14,799 rows | df_baseline: 2,103 rows`

> If the counts look unexpectedly low, check that you are loading the merged  
> master table (not a visit-filtered or phase-filtered subset).


In [ ]:
# ── Replace with your real data loading ──────────────────────────────────────
# df_all      = pd.read_csv("../Data/Final_Data/adni_longitudinal.csv")
# df_baseline = pd.read_csv("../Data/Final_Data/adni_baseline.csv")

# ── Sanity checks ─────────────────────────────────────────────────────────────
# assert "RID"      in df_all.columns and "Years_bl" in df_all.columns, \
#     "df_all must contain RID and Years_bl"
# assert "DX_bl"   in df_baseline.columns, "df_baseline must contain DX_bl"
# print(f"df_all: {len(df_all):,} rows | df_baseline: {len(df_baseline):,} rows")
# print(f"DX_bl counts:\n{df_baseline['DX_bl'].value_counts()}")

print("[PLACEHOLDER] Load df_all and df_baseline here.")


## 2 · MRI harmonisation (ComBat)

ADNI data were collected on **1.5 T and 3 T scanners** across multiple phases
(ADNI1, ADNI2, ADNI3, ADNI-GO). This creates a systematic batch effect in MRI
volumetrics — e.g. Hippocampus volumes measured on a 3 T scanner appear
artificially larger.

**ComBat** (Johnson et al., 2007) removes the scanner-related variance while
preserving biologically meaningful signal (age, sex, diagnosis).

---

### Reading the `plot_before_after` output

The 2 × 2 grid shows histograms of an ICV-normalised MRI feature:

| Panel | Interpretation |
|-------|----------------|
| **Top-left** (Before, by ADNI phase) | Distributions shifted between phases = scanner confound |
| **Top-right** (After, by ADNI phase) | Distributions should overlap more; residual shift is **real biology** (ADNI1 was sicker) |
| **Bottom-left** (Before, by field strength) | Clear 1.5 T vs. 3 T separation |
| **Bottom-right** (After, by field strength) | Gap should be reduced 30–70 % |

A Kruskal-Wallis test is printed after the plot.  
**Residual significance across phases after ComBat is expected and desirable** —
it reflects that ADNI1 enrolled more severe patients, not a scanner artefact.
What we care about is the 1.5 T vs. 3 T gap reduction.


In [ ]:
# df_baseline = run_combat(df_baseline)
# harmonization_report(df_baseline)
# plot_before_after(df_baseline, feature="Hippocampus_ICV")
# plot_before_after(df_baseline, feature="Entorhinal_ICV")
print("ComBat — uncomment once data is loaded.")


## 3 · Feature engineering & cohort assembly

Two cohorts are built independently so each model is specialised for its task:

| Cohort | Transition | Clinical meaning |
|--------|------------|-----------------|
| **MCI → Dementia** | Mild Cognitive Impairment to Dementia | Tracks disease progression in at-risk patients already showing symptoms |
| **CN → MCI** | Cognitively Normal to MCI | Tracks earliest detectable cognitive decline in healthy controls |

### Feature domains

| Domain | Features included |
|--------|-----------------|
| **Demographics** | Age, sex, education, APOE4 carrier status |
| **Cognitive** | MMSE, CDR-SB, ADAS-13, logical memory, RAVLT, FAQ |
| **MRI** | ICV-normalised hippocampus, entorhinal, ventricles, fusiform, mid-temporal, whole-brain |
| **Biomarkers** | CSF Aβ42, tau, p-tau; PET FDG, AV45 |
| **Longitudinal slopes** | Per-subject OLS slope of each feature over pre-cutoff visits — captures *rate of change* |
| **Slope concordance** | Binary flag: both MMSE slope and hippocampus slope are simultaneously negative |

> **Leakage safeguard:** slopes are computed using **only visits before the event/cutoff time**
> for each subject. Post-event visits are never included, preventing the model from
> seeing information that would not have been available at baseline.


In [ ]:
CORE_FEATURES = [
    "AGE", "PTGENDER_num", "PTEDUCAT", "APOE4",
    "MMSE", "CDRSB", "ADAS13", "LDELTOTAL", "RAVLT_immediate", "FAQ",
    "Hippocampus_ICV", "Entorhinal_ICV", "Ventricles_ICV",
    "Fusiform_ICV", "MidTemp_ICV", "WholeBrain_ICV",
    "ABETA", "TAU", "PTAU", "FDG", "AV45",
]

SLOPE_FEATURES = ["MMSE", "CDRSB", "ADAS13", "Hippocampus",
                  "Entorhinal", "Ventricles", "FAQ"]

# ── MCI -> Dementia ───────────────────────────────────────────────────────────
# surv_mci   = build_survival_labels(df_all, df_baseline, "MCI", "Dementia")
# slopes_mci = compute_slopes_cutoff(df_all, surv_mci, SLOPE_FEATURES)
# X_mci, y_ev_mci, y_dur_mci, rids_mci = assemble_cohort(
#     df_baseline, surv_mci, slopes_mci, CORE_FEATURES,
#     [c for c in slopes_mci.columns if c != "RID"])
# X_mci = add_slope_concordance(X_mci)
# print(f"MCI cohort: {len(X_mci)} subjects, {y_ev_mci.mean():.1%} event rate")

# ── CN -> MCI ─────────────────────────────────────────────────────────────────
# surv_cn   = build_survival_labels(df_all, df_baseline, "CN", "MCI")
# slopes_cn = compute_slopes_cutoff(df_all, surv_cn, SLOPE_FEATURES)
# X_cn, y_ev_cn, y_dur_cn, rids_cn = assemble_cohort(
#     df_baseline, surv_cn, slopes_cn, CORE_FEATURES,
#     [c for c in slopes_cn.columns if c != "RID"])
# X_cn = add_slope_concordance(X_cn)
# print(f"CN  cohort: {len(X_cn)} subjects, {y_ev_cn.mean():.1%} event rate")

print("Cohort assembly — uncomment once data is loaded.")


## 4 · Imputation

Missing data is handled in two sequential tiers to minimise the amount of
statistical imputation performed while maximising data completeness.

| Tier | Method | When used |
|------|--------|-----------|
| **Tier 1** | Longitudinal carry-forward within ±1 year | A biomarker was measured at a nearby visit but not at baseline |
| **Tier 2** | MICE (Multiple Imputation by Chained Equations) | Feature is genuinely missing with no nearby observation |

### Why MICE instead of simple mean imputation?

MICE preserves correlations between features (e.g. hippocampus volume and MMSE
are correlated) whereas mean imputation does not. This reduces bias in downstream
model coefficients.

### CSF imputation (optional)

Lumbar puncture is invasive; roughly 40–50 % of ADNI subjects lack CSF Aβ42/tau.
A LightGBM model is trained on subjects with known CSF values using non-invasive
features (MRI, PET, cognition, APOE4) to predict missing values. The holdout
RMSE and R² printed during training indicate imputation quality — R² > 0.4 is
considered reasonable for CSF prediction.


In [ ]:
# ── Tier-1: borrow from nearest visit within ±1 yr ───────────────────────────
# df_all_filled = longitudinal_fill(df_all, SLOPE_FEATURES + CORE_FEATURES)

# ── Tier-2: MICE for remaining NaNs ──────────────────────────────────────────
# X_mci_imp = mice_impute(X_mci)
# X_cn_imp  = mice_impute(X_cn)

# ── Optional: CSF ABETA imputation ───────────────────────────────────────────
# csf_model, csf_cols = build_csf_imputer(df_baseline, target_col="ABETA")
# missing_abeta = X_mci_imp["ABETA"].isna()
# if missing_abeta.any():
#     X_mci_imp.loc[missing_abeta, "ABETA"] = csf_model.predict(
#         X_mci_imp.loc[missing_abeta, csf_cols].fillna(0))
# print(f"Remaining NaNs after imputation: {X_mci_imp.isna().sum().sum()}")

print("Imputation — uncomment once data is loaded.")


## 5 · Holdout split

A **20 % stratified holdout** is removed from both cohorts before any model
training, hyperparameter search, or threshold tuning.

### Why this matters

Without a proper holdout, the reported C-index is optimistic because the model
has indirectly "seen" the test data through hyperparameter selection.
The holdout is the only honest estimate of how the model will perform on new patients.

### How stratification works

Subjects are stratified by `event × duration_quartile` — this ensures the test set
has the same event rate **and** the same follow-up time distribution as the training
set, avoiding a situation where all late events land in training and all early events
land in test.

> **Rule:** the `*_test` variables below are used **exclusively** in Sections 7–9.
> Do not pass them to any training function in Section 6.


In [ ]:
# ── MCI cohort ────────────────────────────────────────────────────────────────
# (
#     X_mci_dev, X_mci_test,
#     y_ev_mci_dev, y_ev_mci_test,
#     y_dur_mci_dev, y_dur_mci_test,
#     idx_mci_dev, idx_mci_test
# ) = make_holdout_split(X_mci_imp, y_ev_mci, y_dur_mci,
#                         test_size=0.20, seed=RANDOM_SEED)

# ── CN cohort ──────────────────────────────────────────────────────────────────
# (
#     X_cn_dev, X_cn_test,
#     y_ev_cn_dev, y_ev_cn_test,
#     y_dur_cn_dev, y_dur_cn_test,
#     idx_cn_dev, idx_cn_test
# ) = make_holdout_split(X_cn_imp, y_ev_cn, y_dur_cn,
#                         test_size=0.20, seed=RANDOM_SEED)

# print(f"MCI dev={len(X_mci_dev)}, test={len(X_mci_test)} | "
#       f"event rate dev={y_ev_mci_dev.mean():.1%}, test={y_ev_mci_test.mean():.1%}")
# print(f"CN  dev={len(X_cn_dev)},  test={len(X_cn_test)}  | "
#       f"event rate dev={y_ev_cn_dev.mean():.1%},  test={y_ev_cn_test.mean():.1%}")

print("Holdout split — uncomment once data is loaded.")


## 6 · Model training

All models receive only the **dev** split. Hyperparameters are chosen via
Optuna Bayesian optimisation using **OOF (out-of-fold) C-index** across 5 folds —
the validation fold is never used to fit the model being evaluated.

### Reading the training output

Each training cell prints three numbers:

```
[MCI->Dementia] Model best OOF C: 0.7812  train-C: 0.8345  gap=+0.0533
```

| Number | Meaning |
|--------|---------|
| `OOF C` | Honest cross-validation estimate — what the model generalises to |
| `train-C` | In-sample score on all dev data — always higher than OOF |
| `gap` | `train-C − OOF C`; a gap > 0.05–0.08 warrants caution about overfitting |

A small gap (≈ 0.02–0.04) is normal and reflects the difference between fitting
and predicting. A large gap (> 0.08) suggests the model is memorising training
patterns rather than learning generalisable ones — consider reducing
`n_estimators` or increasing regularisation.


### 6a · Cox Proportional Hazards (regularised)

**What it does:** fits a linear log-hazard model `h(t|x) = h₀(t) · exp(βᵀx)`.
Each feature gets a single coefficient β that applies uniformly across time
(the proportional hazards assumption).

**Regularisation:** an elastic net penalty (mixture of L1 and L2) shrinks
small coefficients toward zero. The penalty strength (`penalizer`) and
L1/L2 ratio (`l1_ratio`) are tuned by Optuna.

- `l1_ratio = 0` → pure ridge (retains all features, shrinks magnitudes)
- `l1_ratio = 0.5` → elastic net (zeroes out irrelevant features while stabilising correlated ones)

**Expected output:**  
`CoxPH best OOF C: ~0.75–0.80 | penalizer=..., l1_ratio=...`

C-index above 0.75 is clinically strong for this task. Values below 0.65
suggest the linear assumption may not hold well for this feature set.


In [ ]:
COHORT = "MCI->Dementia"

# if RETRAIN:
#     (
#         cox_oof_c, cox_train_c,
#         cox_model, cox_scaler,
#         cox_study
#     ) = run_cox_ph(X_mci_dev, y_ev_mci_dev, y_dur_mci_dev,
#                    label=COHORT, n_trials=30)
#     save_checkpoint("cox_mci", (cox_oof_c, cox_train_c, cox_model, cox_scaler, cox_study))
# else:
#     cox_oof_c, cox_train_c, cox_model, cox_scaler, cox_study = \
#         load_checkpoint("cox_mci")

# print(f"CoxPH: OOF C={cox_oof_c:.4f}, Train C={cox_train_c:.4f}, "
#       f"gap={cox_train_c - cox_oof_c:+.4f}")

print("CoxPH — uncomment once data is loaded.")


### 6b · Gradient Boosting Survival Analysis (GBSA)

**What it does:** fits a gradient boosted ensemble of decision trees to minimise
the Cox partial likelihood (same objective as Cox PH, but with non-linear trees
instead of a linear model). Each tree corrects the residual errors of the
previous ensemble.

**Why it often outperforms Cox PH:** GBSA automatically captures non-linear
relationships (e.g. a threshold effect where only extreme hippocampal atrophy
is predictive) and feature interactions (e.g. APOE4 amplifying the effect of
amyloid burden).

**Key hyperparameters tuned:**

| Parameter | Effect |
|-----------|--------|
| `learning_rate` | Step size per tree; smaller = more trees needed, less overfit |
| `n_estimators` | Number of trees; too many → overfit on training |
| `max_depth` | Tree complexity; depth 1–2 = stumps (high bias), depth 5–6 = complex (high variance) |
| `subsample` | Fraction of subjects used per tree; acts as stochastic regularisation |

**Expected output:**  
`GBSA best OOF Antolini-C: ~0.78–0.84 | in-sample C-td: slightly higher`


In [ ]:
# if RETRAIN:
#     (
#         gbsa_oof_c, gbsa_train_c,
#         gbsa_imp, gbsa_model,
#         gbsa_study, gbsa_oof_preds
#     ) = gbsa_survival_cv(
#         X_mci_dev, y_ev_mci_dev, y_dur_mci_dev,
#         feature_names=X_mci_dev.columns.tolist(),
#         label=COHORT, n_trials=40)
#     save_checkpoint("gbsa_mci", (gbsa_oof_c, gbsa_train_c, gbsa_imp,
#                                   gbsa_model, gbsa_study, gbsa_oof_preds))
# else:
#     (gbsa_oof_c, gbsa_train_c, gbsa_imp, gbsa_model,
#      gbsa_study, gbsa_oof_preds) = load_checkpoint("gbsa_mci")

# print(f"GBSA: OOF C={gbsa_oof_c:.4f}, Train C={gbsa_train_c:.4f}, "
#       f"gap={gbsa_train_c - gbsa_oof_c:+.4f}")

print("GBSA — uncomment once data is loaded.")


### 6c · DeepSurv (neural Cox PH)

**What it does:** replaces the linear `βᵀx` term in Cox PH with a multi-layer
perceptron (MLP), allowing the model to learn non-linear feature representations
before computing the log-hazard. The Cox partial likelihood loss is backpropagated
through the network.

**Regularisation mechanisms used:**

| Mechanism | Purpose |
|-----------|---------|
| Batch normalisation | Stabilises training, acts as implicit regulariser |
| Dropout (tuned 5–50 %) | Randomly zeroes activations during training, prevents co-adaptation |
| Weight decay (L2 on weights) | Keeps weight magnitudes small |
| Early stopping (patience=15) | Halts training when validation loss stops improving |

**Architecture options searched by Optuna:**  
`[32,32]`, `[64,64]`, `[128,128]`, `[256,128,64]`, `[256,256,128]`, `[128,64,32]`

**Expected output:**  
`DeepSurv best OOF C-td: ~0.76–0.83 | arch, dropout, lr, wd printed`

If the gap between `train-C` and `OOF C` is large (> 0.08), increase dropout or
reduce the network width in the architecture options.


In [ ]:
# if RETRAIN:
#     (
#         ds_oof_c, ds_train_c,
#         ds_model, ds_scaler,
#         ds_loss_history, ds_study,
#         ds_oof_preds
#     ) = run_deepsurv(
#         X_mci_dev, y_ev_mci_dev, y_dur_mci_dev,
#         label=COHORT, n_trials=25)
#     save_checkpoint("deepsurv_mci", (ds_oof_c, ds_train_c, ds_model, ds_scaler,
#                                        ds_loss_history, ds_study, ds_oof_preds))
# else:
#     (ds_oof_c, ds_train_c, ds_model, ds_scaler,
#      ds_loss_history, ds_study, ds_oof_preds) = load_checkpoint("deepsurv_mci")

# print(f"DeepSurv: OOF C={ds_oof_c:.4f}, Train C={ds_train_c:.4f}, "
#       f"gap={ds_train_c - ds_oof_c:+.4f}")

print("DeepSurv — uncomment once data is loaded.")


## 7 · Holdout evaluation & bootstrap confidence intervals

**This is the first and only time the test set is used.**  
The C-index computed here is an unbiased estimate of how the models will perform
on future unseen patients.

### Bootstrap CI interpretation

We resample subjects with replacement 1,000 times and recompute the C-index on
each resample using the pre-computed survival curves (no model refit). The 2.5th
and 97.5th percentiles form the 95 % confidence interval.

**How to read the output:**
```
CoxPH  holdout C: 0.7841  95% CI [0.7512, 0.8170]
GBSA   holdout C: 0.8063  95% CI [0.7751, 0.8375]
DS     holdout C: 0.7934  95% CI [0.7599, 0.8249]
```

- A **narrow CI** (width < 0.05) means the estimate is stable; adding more
  patients would not change the ranking much.
- A **wide CI** (width > 0.10) means the estimate is uncertain, often because
  the test set is small or the event rate is low.
- If two CIs **overlap substantially**, the models are not statistically
  distinguishable in performance — report both rather than claiming one is better.


In [ ]:
# ── Evaluate each model on the holdout ───────────────────────────────────────
# cox_test_c,  cox_surv_test  = calc_cox_ph_c(cox_model, cox_scaler,
#                                              X_mci_test, y_ev_mci_test, y_dur_mci_test)
# gbsa_test_c, gbsa_surv_test = calc_gbsa_c(gbsa_model,
#                                            X_mci_test, y_ev_mci_test, y_dur_mci_test)
# ds_test_c,   ds_surv_test   = calc_deepsurv_c(ds_model, ds_scaler,
#                                                X_mci_test, y_ev_mci_test, y_dur_mci_test)

# ── Bootstrap CI (1000 resamples) ─────────────────────────────────────────────
# cox_boot  = bootstrap_cindex_td(y_ev_mci_test, y_dur_mci_test, cox_surv_test)
# gbsa_boot = bootstrap_cindex_td(y_ev_mci_test, y_dur_mci_test, gbsa_surv_test)
# ds_boot   = bootstrap_cindex_td(y_ev_mci_test, y_dur_mci_test, ds_surv_test)

# print(f"CoxPH  holdout C: {cox_boot['point']:.4f}  "
#       f"95% CI [{cox_boot['lower']:.4f}, {cox_boot['upper']:.4f}]")
# print(f"GBSA   holdout C: {gbsa_boot['point']:.4f}  "
#       f"95% CI [{gbsa_boot['lower']:.4f}, {gbsa_boot['upper']:.4f}]")
# print(f"DS     holdout C: {ds_boot['point']:.4f}  "
#       f"95% CI [{ds_boot['lower']:.4f}, {ds_boot['upper']:.4f}]")

print("Holdout evaluation — uncomment once data is loaded.")


## 8 · Ensemble (weighted survival curves, IPCW C-td)

The ensemble blends the three models' **survival curve predictions** (not just
risk scores) using weights optimised by Optuna to maximise the IPCW C-td on
the holdout set.

### Why blend survival curves rather than risk scores?

Blending full `S(t|x)` curves preserves time-resolution — the ensemble can weight
CoxPH heavily at early times (where its linear extrapolation is good) and GBSA
heavily at later times (where it captures non-linear long-term trajectories).

### Reading the ensemble weights output

```
Ensemble weights: {'CoxPH': 0.31, 'GBSA': 0.45, 'DeepSurv': 0.24}
```

- A model receiving **near-zero weight** adds no unique information beyond the
  other two; its predictions are largely redundant.
- A model receiving **> 0.5 weight** dominates the ensemble and likely has the
  most well-calibrated survival curves.
- Uniform weights (~0.33 each) suggest the models are complementary and all
  contribute meaningfully.

### When the ensemble beats individual models

The ensemble typically improves C-index by 0.01–0.03 over the best single model
when the three models make **different types of errors** (one is better on
patients with biomarker data, another on patients with only cognitive scores).
If improvement is negligible, consider that the models are learning very similar
representations from the same features.


In [ ]:
# ens_c, ens_surv, ens_weights = weighted_ensemble_td(
#     {"CoxPH": cox_surv_test,
#      "GBSA":  gbsa_surv_test,
#      "DeepSurv": ds_surv_test},
#     y_ev_mci_test, y_dur_mci_test,
#     label=COHORT, n_trials=50)

# ens_boot = bootstrap_cindex_td(y_ev_mci_test, y_dur_mci_test, ens_surv)
# print(f"Ensemble holdout C: {ens_boot['point']:.4f}  "
#       f"95% CI [{ens_boot['lower']:.4f}, {ens_boot['upper']:.4f}]")
# print(f"Ensemble weights: {ens_weights}")

print("Ensemble — uncomment once data is loaded.")


## 9 · Visualisations

All figures are saved to the `figures/` directory and displayed inline.
Each subsection below produces a specific type of diagnostic plot.
Read the interpretive notes under each heading before drawing conclusions.


### 9a · Feature importance

Three complementary views of feature importance are produced:

**Native importance (GBSA — mean decrease in impurity)**  
How much each feature reduces the Cox loss when chosen as a split point,
averaged across all trees. Fast to compute but biased toward **high-cardinality
continuous features** (e.g. age) even when they contribute little to ranking.

**Cox PH coefficients**  
The signed log-hazard ratio for each feature after standardisation.  
- Positive coefficient → higher value = higher hazard (faster conversion)  
- Negative coefficient → higher value = protective (slower conversion)  
Coefficient magnitude reflects importance *under the proportional hazards assumption*.

**Permutation importance (model-agnostic)**  
Shuffles one feature at a time and measures the resulting C-index drop.
This is the **most reliable importance measure** because it directly quantifies
the effect of removing the feature's information from the model's predictions.

> **When native importance and permutation importance disagree:** trust permutation
> importance. A feature can have high split frequency (native) but low permutation
> importance if the model uses it as a weak tiebreaker that rarely changes the
> final ranking.


In [ ]:
# ── GBSA native importance ─────────────────────────────────────────────────────
# plot_feature_importance(gbsa_imp, model_name="GBSA", cohort=COHORT, top_n=20)

# ── Cox PH signed coefficients ────────────────────────────────────────────────
# plot_cox_coefficients(cox_model, cox_scaler,
#                       feature_names=X_mci_dev.columns.tolist(),
#                       model_name="CoxPH", cohort=COHORT, top_n=20)

# ── Permutation importance ─────────────────────────────────────────────────────
# def cox_predict(X):
#     Xs = pd.DataFrame(cox_scaler.transform(X), columns=X.columns)
#     return -cox_model.predict_log_partial_hazard(Xs).values
# cox_perm = permutation_importance_survival(
#     cox_predict, X_mci_test, y_ev_mci_test, y_dur_mci_test, n_repeats=20)
# plot_permutation_importance(cox_perm, "CoxPH", COHORT)

# def gbsa_predict(X):
#     return gbsa_model.predict(X)
# gbsa_perm = permutation_importance_survival(
#     gbsa_predict, X_mci_test, y_ev_mci_test, y_dur_mci_test, n_repeats=20)
# # Overlay permutation bars on native importance chart:
# plot_feature_importance(gbsa_imp, "GBSA", COHORT, top_n=20, perm_df=gbsa_perm)
# plot_permutation_importance(gbsa_perm, "GBSA", COHORT)

print("Feature importance — uncomment once data is loaded.")


### 9b · Kaplan-Meier curves by predicted risk quartile

The holdout subjects are divided into four equal-sized groups by predicted risk:
Q1 (lowest) to Q4 (highest). A KM curve is drawn for each group.

**What to look for:**
- Well-separated curves → the model discriminates well; Q4 converts much faster than Q1
- Crossing or tightly clustered curves → poor discrimination; the model struggles to
  rank patients correctly
- **Wide confidence bands** → small group sizes (< 20 per quartile); interpret with caution

**Log-rank test p-value** is annotated on each plot.  
- `p < 0.001` → strong evidence that the curves differ (model is discriminating)  
- `p > 0.05` → curves are not statistically distinguishable; model may not be
  clinically useful at this quartile resolution

**Comparing models:** the model whose Q1 and Q4 curves are most widely separated
(largest vertical gap at any time point) has the best discrimination.


In [ ]:
# for model_name, surv_test in [
#     ("CoxPH",    cox_surv_test),
#     ("GBSA",     gbsa_surv_test),
#     ("DeepSurv", ds_surv_test),
#     ("Ensemble", ens_surv),
# ]:
#     risk_scores = 1.0 - surv_test.values.mean(axis=0)
#     km_risk_quartile(risk_scores, y_ev_mci_test, y_dur_mci_test,
#                      model_name=model_name, cohort=COHORT, log_rank=True)

print("KM quartile curves — uncomment once data is loaded.")


### 9c · Individual survival curve grids

A random sample of 12 subjects from the holdout set, each showing their
predicted survival curve `S(t|x)` from one model.

**Colour coding:**
- 🔴 Red panels — subject experienced the event (conversion)
- 🔵 Blue panels — subject was censored (did not convert in follow-up)

**Vertical dashed line** marks the subject's observed exit time.

**What a well-behaved model looks like:**
- Red panels should have steep, rapidly declining curves
- Blue panels should have flatter, slowly declining curves
- At the exit time, `S(t)` for event subjects should generally be lower than
  for censored subjects — the model should have "seen it coming"

**Red flags:**
- A censored subject with a steeply declining curve was assigned high risk
  but never converted — this is not necessarily wrong (they may have converted
  after follow-up ended)
- An event subject with a flat curve — the model failed to identify this
  high-risk patient; these are the model's blind spots


In [ ]:
# for model_name, surv_test in [
#     ("CoxPH",    cox_surv_test),
#     ("GBSA",     gbsa_surv_test),
#     ("DeepSurv", ds_surv_test),
# ]:
#     plot_survival_curve_grid(surv_test, y_ev_mci_test, y_dur_mci_test,
#                              model_name=model_name, cohort=COHORT, n_subjects=12)

print("Survival grids — uncomment once data is loaded.")


### 9d · Mean predicted survival vs. empirical KM: event vs. stable

The left panel shows the **average predicted** `S(t|x)` for converters vs.
stable MCI subjects, with an IQR band. The right panel shows the raw KM estimate
for the same two groups.

**How to use this plot:**
- If the predicted curves (left) track the KM curves (right) closely, the model is
  **well-calibrated** — the predicted probabilities match observed rates.
- If the predicted curves diverge substantially from KM, the model is either
  over-predicting or under-predicting risk, even if its C-index is high.
  *(A model can rank patients correctly but assign the wrong absolute probabilities.)*
- The **gap between the two groups** should be similar in both panels; a much
  larger predicted gap than observed KM gap means the model exaggerates risk differences.


In [ ]:
# event_mask = y_ev_mci_test.astype(bool)
# for model_name, surv_test in [
#     ("CoxPH",    cox_surv_test),
#     ("GBSA",     gbsa_surv_test),
#     ("DeepSurv", ds_surv_test),
# ]:
#     plot_median_survival_by_group(
#         surv_test, y_ev_mci_test, y_dur_mci_test,
#         group_mask=event_mask,
#         group_labels=("Stable MCI", "Converter"),
#         title=f"{model_name}: Mean Predicted Survival by Event Status",
#         fname_stem=model_name.lower().replace(" ", "_"),
#         cohort=COHORT,
#     )

print("Mean predicted vs KM — uncomment once data is loaded.")


### 9e · KM curves by APOE4 carrier status

APOE4 is the strongest known genetic risk factor for late-onset Alzheimer's disease.
Carriers have approximately 3× higher risk of conversion from MCI to dementia
compared to non-carriers.

**What to look for:**
- Strong separation between carriers and non-carriers validates that the cohort
  is behaving as expected biologically
- The log-rank p-value should be **highly significant** (p < 0.001) if sample
  size is adequate
- If APOE4 carriers show little difference, check for data quality issues or
  a heavily pre-selected cohort (e.g. only EMCI patients who are less likely to convert)

This plot is a **biological sanity check** — if APOE4 does not separate the curves,
investigate the data before trusting model outputs.


In [ ]:
# if "APOE4" in X_mci_test.columns:
#     apoe4_mask = (X_mci_test["APOE4"].values > 0)
#     km_binary_comparison(
#         y_ev_mci_test, y_dur_mci_test,
#         group_mask=apoe4_mask,
#         group_labels=("APOE4 non-carrier", "APOE4 carrier"),
#         title="KM: APOE4 Carrier Status [MCI->Dementia]",
#         fname_stem="km_apoe4",
#         cohort=COHORT,
#     )

print("APOE4 KM — uncomment once data is loaded.")


### 9f · ROC curves at fixed time horizons (3-year, 5-year)

ROC at a fixed horizon converts the survival model into a binary classifier:
"will this patient convert within *t* years?" Subjects censored before the
horizon are excluded (their outcome is unknown).

**Reading the plot:**
- **AUC = 1.0** → perfect discrimination at that horizon
- **AUC = 0.5** → random (the model provides no information for this horizon)
- AUC at **3 years** and **5 years** may differ substantially — some models
  are better at short-term vs. long-term prediction

**Clinical relevance:**
- 3-year AUC is most relevant for **treatment trial recruitment** (patient
  needs to convert during trial period)
- 5-year AUC is most relevant for **long-term clinical counselling**

If 3-year AUC is high but 5-year AUC is lower, the model is better at
identifying patients who are close to converting than those who have many
years of stability ahead.


In [ ]:
# for model_name, surv_test in [
#     ("CoxPH",    cox_surv_test),
#     ("GBSA",     gbsa_surv_test),
#     ("DeepSurv", ds_surv_test),
#     ("Ensemble", ens_surv),
# ]:
#     plot_roc_horizons(surv_test, y_ev_mci_test, y_dur_mci_test,
#                       model_name=model_name, cohort=COHORT,
#                       horizons=HORIZONS)

print("ROC horizons — uncomment once data is loaded.")


### 9g · Time-dependent Brier score (IPCW)

The Brier score at time `t` is the mean squared error between the model's
predicted `S(t|x)` and the true binary outcome at `t`. Lower is better.

**Reference points:**
| Score | Meaning |
|-------|---------|
| **0.00** | Perfect predictions at every time point |
| **0.25** | Uninformative model (predicting 0.5 for everyone) |
| **> 0.25** | Worse than a naive constant predictor — something is wrong |

The **Integrated Brier Score (IBS)** is the area under the Brier curve
normalised by the time range, printed and annotated on the plot.

**What to look for:**
- The Brier score typically starts near 0 (few events early), rises as events
  accumulate, then stabilises when follow-up ends and subjects are censored
- A model with a **lower IBS** than others is more accurate on average across all
  time points — this is a stricter test than C-index because it evaluates
  *calibration* (correct probabilities), not just *discrimination* (correct ranking)
- If the Brier score rises sharply after year 5, the model may be
  extrapolating poorly beyond the range of training data

**A model can have a high C-index but a poor Brier score** — this means it ranks
patients correctly but assigns wrong absolute probabilities. For clinical
decision-making, calibration matters as much as discrimination.


In [ ]:
# ibs_results = {}
# for model_name, surv_test in [
#     ("CoxPH",    cox_surv_test),
#     ("GBSA",     gbsa_surv_test),
#     ("DeepSurv", ds_surv_test),
#     ("Ensemble", ens_surv),
# ]:
#     ibs = plot_brier_score(surv_test, y_ev_mci_test, y_dur_mci_test,
#                            model_name=model_name, cohort=COHORT)
#     ibs_results[model_name] = ibs

# print("\nIntegrated Brier Scores (lower is better):")
# for m, ibs in sorted(ibs_results.items(), key=lambda x: x[1]):
#     print(f"  {m:12s}: {ibs:.4f}")

print("Brier score — uncomment once data is loaded.")


### 9h · Calibration plots at 3-year and 5-year horizons

Calibration asks: *"When the model predicts a 40 % chance of conversion by
year 3, do roughly 40 % of those patients actually convert?"*

**How it is computed:**
1. Subjects are divided into bins by predicted risk (e.g. quintiles)
2. For each bin, the **average predicted probability** is compared to the
   **observed KM event rate** (with 95 % CI shown as error bars)
3. A perfectly calibrated model falls on the diagonal `y = x`

**Reading the plot:**
- Points **above the diagonal** → model **under-predicts** risk (actual rate is
  higher than predicted; the model is over-confident that patients will stay stable)
- Points **below the diagonal** → model **over-predicts** risk (actual rate is
  lower than predicted; the model alarms too many stable patients)
- **Tight error bars** → reliable KM estimates in that bin (many subjects)
- **Wide error bars** → few subjects in the bin; the observed rate is noisy

**Clinical implication:**  
A well-discriminating model (high C-index) that is poorly calibrated may not
be suitable for risk communication to patients — e.g. telling a patient they
have a "60 % chance of converting in 3 years" when the true rate is 30 % is
harmful. Calibration plots reveal this directly.


In [ ]:
# for h in HORIZONS:
#     for model_name, surv_test in [
#         ("CoxPH",    cox_surv_test),
#         ("GBSA",     gbsa_surv_test),
#         ("DeepSurv", ds_surv_test),
#         ("Ensemble", ens_surv),
#     ]:
#         plot_calibration_horizon(surv_test, y_ev_mci_test, y_dur_mci_test,
#                                  model_name=model_name, cohort=COHORT,
#                                  horizon=h, n_bins=5)

print("Calibration — uncomment once data is loaded.")


### 9i · Overfitting summary (OOF vs. train vs. holdout)

This grouped bar chart places all three C-index estimates side-by-side for
every model, making it easy to spot which models overfit most severely.

| Bar colour | Estimate | What it tells us |
|------------|----------|-----------------|
| 🔵 Blue | OOF CV C-index | Honest estimate from training-time cross-validation |
| 🟠 Orange | In-sample (train) C-index | How well the model fits its own training data |
| 🟢 Green | Holdout test C-index | **True** out-of-sample performance on unseen data |

**Healthy pattern:** Blue ≈ Green (OOF predicts holdout well), Orange slightly above Blue.

**Overfit pattern:** Orange >> Blue ≈ Green (model memorised training data;
OOF successfully flagged the problem before holdout evaluation).

**Leakage pattern:** Green >> Blue (test set was somehow contaminated, or the
OOF estimate was pessimistic due to a small dev set). Double-check the holdout
split was done correctly if Green is substantially higher than Blue.


In [ ]:
# results_dict = {
#     "CoxPH":    {"oof_c": cox_oof_c,  "train_c": cox_train_c,  "test_c": cox_boot["point"]},
#     "GBSA":     {"oof_c": gbsa_oof_c, "train_c": gbsa_train_c, "test_c": gbsa_boot["point"]},
#     "DeepSurv": {"oof_c": ds_oof_c,   "train_c": ds_train_c,   "test_c": ds_boot["point"]},
#     "Ensemble": {"oof_c": float("nan"), "train_c": float("nan"), "test_c": ens_boot["point"]},
# }
# plot_overfitting_summary(results_dict, cohort=COHORT)

print("Overfitting summary — uncomment once data is loaded.")


### 9j · Bootstrap CI comparison across models

Two complementary plots are produced:

**Violin + box plot (`plot_bootstrap_ci`)**  
Shows the full bootstrap *distribution* of holdout C-index for each model.
The violin width at any value represents how often that C-index appeared across
1,000 bootstrap resamples. The diamond marker is the point estimate; whiskers
show the 95 % CI.

- A **narrow violin** means the estimate is stable — resampling subjects barely
  changes the C-index
- A **wide or bimodal violin** means the estimate is sensitive to which subjects
  are in the test set; the model's performance varies a lot depending on patient
  characteristics

**Horizontal bar chart (`plot_model_comparison`)**  
Orders models from worst to best with CI error bars. The annotation shows
`point estimate [lower, upper]`.

**How to compare two models:**  
If their 95 % CIs **do not overlap** at all, the difference is statistically
significant (at roughly the 5 % level for a two-sample comparison).
Overlapping CIs do not mean the models are equivalent — they mean we cannot
rule out equivalence with this sample size.


In [ ]:
# boot_results = {
#     "CoxPH":    cox_boot,
#     "GBSA":     gbsa_boot,
#     "DeepSurv": ds_boot,
#     "Ensemble": ens_boot,
# }
# plot_bootstrap_ci(boot_results, cohort=COHORT)
# plot_model_comparison(boot_results, cohort=COHORT)

# # Per-model bootstrap histograms (detailed view)
# for m, b in boot_results.items():
#     plot_bootstrap_distribution(b, model_name=m, cohort=COHORT)

print("Bootstrap CI — uncomment once data is loaded.")


### 9k · DeepSurv learning curves

Per-epoch training and validation loss for the final DeepSurv model,
measured as negative partial log-likelihood (lower = better fit).

**What a healthy curve looks like:**
- Both train and val loss decrease in early epochs
- Val loss levels off or improves slightly more slowly than train
- The model stops (via early stopping) near the val loss minimum

**Overfitting signature:**
- Train loss continues decreasing while val loss starts **increasing** — this is
  the classic overfit pattern; the model is memorising training examples
- If this is seen, the early stopping patience may need to be reduced, or
  dropout / weight decay increased

**Underfitting signature:**
- Both curves plateau high and early — the model is too small or learning rate
  is too high; try a larger architecture or lower learning rate

**Note:** if the loss history is empty, the pycox training log format was not
compatible with the extraction code. Check `run_deepsurv` in `modeling.py` and
inspect the `log` object returned by `final.fit()`.


In [ ]:
# plot_learning_curve_deepsurv(ds_loss_history, model_name="DeepSurv", cohort=COHORT)

print("Learning curves — uncomment once data is loaded.")


### 9l · Optuna HPO history

Scatter plot of every Optuna trial's OOF C-index, with the running best
value highlighted as a step curve.

**What to look for:**
- A **rising best-so-far curve** that flattens after ~20–30 trials means the
  search converged — more trials would not meaningfully improve performance
- A best-so-far curve that is **still rising** at the last trial means
  increasing `n_trials` could yield a better model
- A **highly scattered** point cloud (large variance between trials) means
  the hyperparameter landscape is noisy; individual trial results are
  unreliable, but the best-of-many is still a good estimate
- A narrow, high-concentration band of trial values means most hyperparameter
  combinations work similarly well — the model is robust to HPO choices


In [ ]:
# for study, name in [(cox_study, "CoxPH"), (gbsa_study, "GBSA"), (ds_study, "DeepSurv")]:
#     plot_optuna_history(study, model_name=name, cohort=COHORT)

print("Optuna history — uncomment once data is loaded.")


## 10 · CN → MCI cohort

Repeat Sections 6–9 for the CN → MCI transition using the `*_cn_*` variables
created in Section 5.

### How this cohort differs from MCI → Dementia

| Aspect | MCI → Dementia | CN → MCI |
|--------|---------------|---------|
| **Event rate** | Higher (~50–60 %) | Lower (~25–35 %) |
| **Follow-up needed** | Shorter (conversion faster) | Longer (years to first signs) |
| **Most predictive features** | Cognitive scores, hippocampus | APOE4, amyloid PET, subtle cognitive change |
| **Clinical use case** | Trial enrichment, near-term care planning | Prevention trial screening |

Because the CN → MCI event rate is lower, expect wider bootstrap CIs and
potentially lower C-index values. A C-index of 0.70 on CN → MCI is clinically
comparable to 0.80 on MCI → Dementia given the harder prediction task.


In [ ]:
COHORT_CN = "CN->MCI"

# ── Repeat training (Sections 6a–6c) with X_cn_dev, y_ev_cn_dev, y_dur_cn_dev ─
# ── Repeat evaluation (Sections 7–9) with X_cn_test, y_ev_cn_test, y_dur_cn_test

print("CN->MCI cohort — mirror sections 6–9 here.")


## 11 · Save results

Saves a summary CSV to `outputs/` with one row per model containing:

| Column | Description |
|--------|-------------|
| `model` | Model name |
| `oof_c` | Cross-validation C-index (training-time estimate) |
| `train_c` | In-sample C-index (overfitting diagnostic) |
| `test_c` | Holdout C-index (primary reported metric) |
| `ci_lo`, `ci_hi` | 95 % bootstrap CI bounds on holdout C |


In [ ]:
# pd.DataFrame({
#     "model":   ["CoxPH",       "GBSA",        "DeepSurv",   "Ensemble"],
#     "oof_c":   [cox_oof_c,     gbsa_oof_c,    ds_oof_c,     float("nan")],
#     "train_c": [cox_train_c,   gbsa_train_c,  ds_train_c,   float("nan")],
#     "test_c":  [cox_boot["point"], gbsa_boot["point"],
#                 ds_boot["point"],  ens_boot["point"]],
#     "ci_lo":   [cox_boot["lower"], gbsa_boot["lower"],
#                 ds_boot["lower"],  ens_boot["lower"]],
#     "ci_hi":   [cox_boot["upper"], gbsa_boot["upper"],
#                 ds_boot["upper"],  ens_boot["upper"]],
# }).to_csv(OUT_DIR / "mci_model_results.csv", index=False)
# print(f"Results saved to {OUT_DIR / 'mci_model_results.csv'}")

print("Save results — uncomment once models are trained.")
